# Experiment 5: Pipeline TF-IDF Parameter Tuning with MLflow

Tuning TF-IDF vectorizer parameters (max_features, ngram_range, min_df, max_df) in a scikit-learn Pipeline with Logistic Regression, logging all trials and models to MLflow & DagsHub.

In [ ]:
import os
import re
import string
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import mlflow
import mlflow.sklearn
import dagshub

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

## 1. Setup MLflow Tracking

In [ ]:
DATA_PATH = '../data/balanced_sentiment_dataset.csv' if os.path.exists('../data/balanced_sentiment_dataset.csv') else 'data/balanced_sentiment_dataset.csv'

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://dagshub.com/rohitkr8527/sentiment-analysis.mlflow")
REPO_OWNER = os.getenv("DAGSHUB_REPO_OWNER", "rohitkr8527")
REPO_NAME = os.getenv("DAGSHUB_REPO_NAME", "sentiment-analysis")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
dagshub_token = os.getenv("DAGSHUB_TOKEN")
if dagshub_token:
    os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_token
    os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token
    dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)

mlflow.set_experiment("Tf-Idf Hyperparameter Tuning")
print("MLflow experiment configured: 'Tf-Idf Hyperparameter Tuning'")

## 2. Load and Preprocess Data

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    words = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return " ".join(words)

df = pd.read_csv(DATA_PATH)
df["text"] = df["text"].astype(str).apply(preprocess_text)

X_train, X_test, y_train, y_test = train_test_split(df["text"], df["sentiment"], test_size=0.2, random_state=42)
print(f"Total training texts: {len(X_train)}, test texts: {len(X_test)}")

## 3. Pipeline GridSearch and MLflow Trial Logging

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(
        solver="saga",
        penalty="elasticnet",
        l1_ratio=0.5,
        C=1.0,
        max_iter=500
    ))
])

param_grid = {
    "tfidf__max_df": [0.9, 1.0],
    "tfidf__min_df": [1, 5],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__max_features": [5000, 10000]
}

with mlflow.start_run(run_name="GridSearchCV_TFIDF_Tuning") as parent_run:
    grid_search = GridSearchCV(
        pipeline,
        param_grid=param_grid,
        cv=3,
        scoring="f1",
        n_jobs=-1,
        verbose=1
    )
    grid_search.fit(X_train, y_train)

    # Log each trial as a nested MLflow child run
    for i, (params, mean_score, std_score) in enumerate(zip(
        grid_search.cv_results_["params"],
        grid_search.cv_results_["mean_test_score"],
        grid_search.cv_results_["std_test_score"]
    )):
        with mlflow.start_run(run_name=f"TFIDF_trial_{i+1}", nested=True):
            pipeline.set_params(**params)
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)

            metrics = {
                "accuracy": float(accuracy_score(y_test, y_pred)),
                "precision": float(precision_score(y_test, y_pred)),
                "recall": float(recall_score(y_test, y_pred)),
                "f1_score": float(f1_score(y_test, y_pred)),
                "mean_cv_f1": float(mean_score),
                "std_cv_f1": float(std_score)
            }

            mlflow.log_params(params)
            mlflow.log_params({
                "clf__solver": "saga",
                "clf__penalty": "elasticnet",
                "clf__C": 1.0,
                "clf__l1_ratio": 0.5
            })
            mlflow.log_metrics(metrics)
            print(f"Trial {i+1}: {params} -> Accuracy: {metrics['accuracy']:.4f}, F1: {metrics['f1_score']:.4f}")

    # Best Pipeline Artifact & Params Logging
    best_params = grid_search.best_params_
    best_pipeline = grid_search.best_estimator_
    best_f1 = float(grid_search.best_score_)

    mlflow.log_params(best_params)
    mlflow.log_metric("best_cv_f1_score", best_f1)
    mlflow.sklearn.log_model(best_pipeline, "best_pipeline_model")

    print(f"\n--- Best TF-IDF Parameters: {best_params} ---")
    print(f"Best CV F1-Score: {best_f1:.4f}")